# newsrec_bench — Kaggle runnable notebook

So sánh **NRMS / NAML / Fastformer / CAUM / LightGCN (graph) / LLM-encoder / các cải tiến** trên **cùng loss + metric**, đo **chất lượng (AUC/MRR/nDCG) và tốc độ**.

## Cách dùng (3 bước)
1. **Settings → Accelerator = GPU** (P100/T4). Muốn dùng embedding BGE thật thì **Internet = On**.
2. (Tùy chọn) **Add Input → Datasets** dataset MIND (vd `arashnic/mind-news-dataset`), rồi điền `MIND_TRAIN`/`MIND_DEV` ở ô Config. **Không gắn cũng chạy được** (tự chạy dữ liệu synthetic demo).
3. **Run All**. Kết quả in ra + lưu `results/`.

> Notebook này TỰ CHỨA: các ô `%%writefile` sẽ ghi ra các module `.py` rồi chạy. Không cần clone repo.

### 0. Kiểm tra môi trường

In [ ]:
import torch, numpy, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

### 1. Ghi các module (self-contained)

In [ ]:
%%writefile metrics.py
"""Ranking metrics for news recommendation, computed per-impression.

Pure NumPy so there is no dependency beyond numpy. Definitions match the
Microsoft Recommenders / MIND leaderboard conventions (MRR, nDCG@k) plus a
tie-aware AUC.
"""
from __future__ import annotations

import numpy as np


def auc_score(labels: np.ndarray, scores: np.ndarray) -> float | None:
    """Area under ROC curve for one impression (tie-aware, rank based).

    Returns None if the impression has no positive or no negative (undefined),
    so the caller can skip it in the mean.
    """
    labels = np.asarray(labels)
    scores = np.asarray(scores, dtype=np.float64)
    n_pos = int(labels.sum())
    n_neg = int(len(labels) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return None
    # Average ranks (1..n), ties share the mean rank -> Mann-Whitney U form.
    _, inv, counts = np.unique(scores, return_inverse=True, return_counts=True)
    cum = np.cumsum(counts)
    start = cum - counts
    avg = (start + cum + 1) / 2.0  # average 1-based rank per distinct value
    ranks = avg[inv]
    sum_ranks_pos = ranks[labels == 1].sum()
    return (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def mrr_score(labels: np.ndarray, scores: np.ndarray) -> float:
    order = np.argsort(scores)[::-1]
    y = np.asarray(labels)[order]
    rr = y / (np.arange(len(y)) + 1)
    denom = y.sum()
    return float(rr.sum() / denom) if denom > 0 else 0.0


def _dcg(labels: np.ndarray, scores: np.ndarray, k: int) -> float:
    order = np.argsort(scores)[::-1][:k]
    gains = (2 ** np.asarray(labels)[order] - 1).astype(np.float64)
    discounts = np.log2(np.arange(len(gains)) + 2)
    return float((gains / discounts).sum())


def ndcg_score(labels: np.ndarray, scores: np.ndarray, k: int) -> float:
    labels = np.asarray(labels)
    ideal = _dcg(labels, labels, k)
    if ideal == 0:
        return 0.0
    return _dcg(labels, scores, k) / ideal


def aggregate(impressions: list[tuple[np.ndarray, np.ndarray]]) -> dict[str, float]:
    """Mean metrics over a list of (labels, scores) impressions."""
    aucs, mrrs, n5, n10 = [], [], [], []
    for labels, scores in impressions:
        a = auc_score(labels, scores)
        if a is not None:
            aucs.append(a)
        mrrs.append(mrr_score(labels, scores))
        n5.append(ndcg_score(labels, scores, 5))
        n10.append(ndcg_score(labels, scores, 10))
    return {
        "auc": float(np.mean(aucs)) if aucs else 0.0,
        "mrr": float(np.mean(mrrs)) if mrrs else 0.0,
        "ndcg@5": float(np.mean(n5)) if n5 else 0.0,
        "ndcg@10": float(np.mean(n10)) if n10 else 0.0,
    }


In [ ]:
%%writefile data.py
"""Data layer for the news-recommendation benchmark.

Two sources, one interface:

* ``MindData.from_mind(train_dir, dev_dir)`` reads the real MIND dataset
  (``news.tsv`` + ``behaviors.tsv`` in each split; download from
  https://msnews.github.io). Nothing about the models below is MIND-specific.
* ``MindData.synthetic(...)`` generates a MIND-shaped dataset with a *known*
  generative process so every model family has something real to learn:
    - a per-news topic distribution drives the words in its title  -> content
      encoders (NRMS/NAML/Fastformer/LLM) can recover topic affinity;
    - a low-rank user/news latent factor adds collaborative signal not visible
      from text  -> the pure-graph model (LightGCN) can exploit it;
    - a frozen "LLM" embedding is a noisy linear image of the topic vector,
      standing in for a precomputed sentence-transformer / LLM encoding.
  This makes the quality comparison *meaningful as a controlled demo*; absolute
  numbers are not comparable to real-MIND leaderboards.

Everything is returned as plain tensors + index tables so the training loop in
``bench.py`` is identical for all models.
"""
from __future__ import annotations

import random
from dataclasses import dataclass, field

import numpy as np
import torch

PAD = 0  # reserved word / padding index


@dataclass
class Batch:
    hist_title: torch.Tensor  # [B, H, L] long
    hist_cat: torch.Tensor    # [B, H]    long
    hist_idx: torch.Tensor    # [B, H]    long (global news index)
    hist_mask: torch.Tensor   # [B, H]    bool (True = real click)
    cand_title: torch.Tensor  # [B, C, L] long
    cand_cat: torch.Tensor    # [B, C]    long
    cand_idx: torch.Tensor    # [B, C]    long
    cand_mask: torch.Tensor   # [B, C]    bool (True = real candidate; eval padding)
    user_idx: torch.Tensor    # [B]       long
    labels: torch.Tensor      # [B, C]    float (1 = clicked) — used for eval

    def to(self, device):
        for f in self.__dataclass_fields__:
            setattr(self, f, getattr(self, f).to(device))
        return self


@dataclass
class MindData:
    # news feature tables, indexed by global news id (0 = pad news)
    news_title: np.ndarray            # [n_news, L] int
    news_cat: np.ndarray              # [n_news]    int
    n_news: int
    n_users: int
    vocab_size: int
    n_cat: int
    title_len: int
    max_hist: int
    llm_emb: np.ndarray               # [n_news, llm_dim] float32 (frozen)
    # samples
    train: list = field(default_factory=list)  # (user, hist[list], pos, [neg...])
    dev: list = field(default_factory=list)     # (user, hist[list], cands[list], labels[list])
    # graph adjacency (built lazily from train interactions)
    _edges: list = field(default_factory=list)  # (user, news) pairs seen in training

    # ---------------------------------------------------------------- batching
    def _pad_hist(self, hist: list[int]):
        hist = hist[-self.max_hist:]
        mask = [True] * len(hist)
        while len(hist) < self.max_hist:
            hist.append(PAD)
            mask.append(False)
        return hist, mask

    def _gather(self, idxs: list[int]):
        """Title + category rows for a list of news ids."""
        title = self.news_title[idxs]                 # [n, L]
        cat = self.news_cat[idxs]                     # [n]
        return title, cat

    def collate_train(self, rows) -> Batch:
        H, L = self.max_hist, self.title_len
        C = 1 + len(rows[0][3])  # 1 positive + n_neg
        B = len(rows)
        hist_title = np.zeros((B, H, L), np.int64)
        hist_cat = np.zeros((B, H), np.int64)
        hist_idx = np.zeros((B, H), np.int64)
        hist_mask = np.zeros((B, H), bool)
        cand_title = np.zeros((B, C, L), np.int64)
        cand_cat = np.zeros((B, C), np.int64)
        cand_idx = np.zeros((B, C), np.int64)
        user_idx = np.zeros((B,), np.int64)
        for b, (user, hist, pos, negs) in enumerate(rows):
            h, m = self._pad_hist(list(hist))
            hist_idx[b] = h
            hist_mask[b] = m
            hist_title[b], hist_cat[b] = self._gather(h)
            cands = [pos] + list(negs)
            cand_idx[b] = cands
            cand_title[b], cand_cat[b] = self._gather(cands)
            user_idx[b] = user
        cand_mask = np.ones((B, C), bool)
        labels = np.zeros((B, C), np.float32)
        labels[:, 0] = 1.0  # positive is always slot 0 during training
        return _to_batch(hist_title, hist_cat, hist_idx, hist_mask,
                         cand_title, cand_cat, cand_idx, cand_mask, user_idx, labels)

    def collate_eval(self, rows) -> Batch:
        """Eval impressions have variable candidate counts -> pad to max C."""
        H, L = self.max_hist, self.title_len
        C = max(len(r[2]) for r in rows)
        B = len(rows)
        hist_title = np.zeros((B, H, L), np.int64)
        hist_cat = np.zeros((B, H), np.int64)
        hist_idx = np.zeros((B, H), np.int64)
        hist_mask = np.zeros((B, H), bool)
        cand_title = np.zeros((B, C, L), np.int64)
        cand_cat = np.zeros((B, C), np.int64)
        cand_idx = np.zeros((B, C), np.int64)
        cand_mask = np.zeros((B, C), bool)
        labels = np.zeros((B, C), np.float32)
        user_idx = np.zeros((B,), np.int64)
        for b, (user, hist, cands, labs) in enumerate(rows):
            h, m = self._pad_hist(list(hist))
            hist_idx[b] = h
            hist_mask[b] = m
            hist_title[b], hist_cat[b] = self._gather(h)
            n = len(cands)
            cand_idx[b, :n] = cands
            t, c = self._gather(cands)
            cand_title[b, :n] = t
            cand_cat[b, :n] = c
            cand_mask[b, :n] = True
            labels[b, :n] = labs
            user_idx[b] = user
        return _to_batch(hist_title, hist_cat, hist_idx, hist_mask,
                         cand_title, cand_cat, cand_idx, cand_mask, user_idx, labels)

    # --------------------------------------------------------------- graph adj
    def build_adjacency(self) -> torch.Tensor:
        """Symmetric normalized bipartite adjacency for LightGCN.

        Node order: users [0, n_users) then news [n_users, n_users + n_news).
        Returns a sparse FloatTensor D^-1/2 (A) D^-1/2.
        """
        U, N = self.n_users, self.n_news
        rows, cols = [], []
        for u, n in self._edges:
            rows += [u, U + n]
            cols += [U + n, u]
        if not rows:  # degenerate guard
            rows, cols = [0], [0]
        idx = np.array([rows, cols], dtype=np.int64)
        vals = np.ones(idx.shape[1], np.float32)
        size = U + N
        deg = np.zeros(size, np.float64)
        np.add.at(deg, idx[0], vals)
        dinv = np.zeros(size, np.float64)
        nz = deg > 0
        dinv[nz] = deg[nz] ** -0.5
        norm = (dinv[idx[0]] * dinv[idx[1]]).astype(np.float32)
        return torch.sparse_coo_tensor(
            torch.from_numpy(idx), torch.from_numpy(norm), (size, size)
        ).coalesce()

    # -------------------------------------------------------------- factories
    @classmethod
    def synthetic(
        cls,
        n_users: int = 2000,
        n_news: int = 3000,
        n_topics: int = 16,
        words_per_topic: int = 50,
        title_len: int = 12,
        max_hist: int = 30,
        n_train: int = 8000,
        n_dev: int = 2000,
        n_neg: int = 4,
        cands_per_impr: int = 20,
        collab_dim: int = 8,
        llm_dim: int = 64,
        seed: int = 0,
    ) -> "MindData":
        rng = np.random.default_rng(seed)
        vocab_size = 1 + n_topics * words_per_topic  # +1 for PAD

        # --- latent structure -------------------------------------------------
        # Each news has ONE clear primary topic (its title words + category + LLM
        # embedding all signal it), so content models have a strong, recoverable
        # signal. Users have a PEAKED preference over topics.
        prim = rng.integers(0, n_topics, size=n_news)            # primary topic id
        news_topic = np.full((n_news, n_topics), 0.1 / n_topics)  # small background
        news_topic[np.arange(n_news), prim] += 0.9
        news_topic /= news_topic.sum(axis=1, keepdims=True)
        news_topic[PAD] = 0
        news_cat = prim.astype(np.int64)                         # category == topic
        news_cat[PAD] = 0
        logits = rng.normal(size=(n_users, n_topics)) * 4.0      # peaked user prefs
        user_pref = np.exp(logits - logits.max(1, keepdims=True))
        user_pref /= user_pref.sum(axis=1, keepdims=True)
        # collaborative low-rank factors (signal NOT explainable from text)
        user_fac = rng.normal(scale=1.0, size=(n_users, collab_dim))
        news_fac = rng.normal(scale=1.0, size=(n_news, collab_dim))

        # --- titles: sample words from the news' topic mixture ----------------
        news_title = np.zeros((n_news, title_len), np.int64)
        for n in range(1, n_news):
            topics = rng.choice(n_topics, size=title_len, p=news_topic[n])
            offs = rng.integers(0, words_per_topic, size=title_len)
            news_title[n] = 1 + topics * words_per_topic + offs

        # --- frozen "LLM" embedding: noisy linear image of the topic vector ---
        proj = rng.normal(scale=1.0, size=(n_topics, llm_dim))
        llm_emb = news_topic @ proj + rng.normal(scale=0.1, size=(n_news, llm_dim))
        llm_emb = (llm_emb / (np.linalg.norm(llm_emb, axis=1, keepdims=True) + 1e-8)).astype(np.float32)

        # Clicks are driven mainly by TOPIC: a user clicks news in their few liked
        # topics with high probability and off-topic news rarely — a strong,
        # content-learnable ranking signal. A small collaborative bump (from the
        # low-rank factors) leaves a little extra structure for the graph model.
        liked = [set(np.argsort(user_pref[u])[-2:].tolist()) for u in range(n_users)]

        def click_prob(u, n):
            base = 0.85 if prim[n] in liked[u] else 0.06
            collab = float(user_fac[u] @ news_fac[n]) / np.sqrt(collab_dim)
            return min(0.95, max(0.02, base + 0.08 * collab))

        # --- per-user click history: news from the user's liked topics --------
        hist_by_user: list[list[int]] = [[] for _ in range(n_users)]
        edges: list[tuple[int, int]] = []
        for u in range(n_users):
            cand = rng.integers(1, n_news, size=120)
            keep = [int(n) for n in cand if prim[n] in liked[u]][:max_hist]
            hist_by_user[u] = keep
            edges += [(u, n) for n in keep]

        def make_impression(u, rng):
            hist = hist_by_user[u]
            pool = rng.integers(1, n_news, size=cands_per_impr * 3)
            pool = [int(n) for n in pool if n not in hist]
            clicks = np.array([rng.random() < click_prob(u, n) for n in pool])
            return hist, pool, clicks

        # --- train samples (1 pos + n_neg, softmax-over-candidates style) ------
        train = []
        tries = 0
        while len(train) < n_train and tries < n_train * 20:
            tries += 1
            u = int(rng.integers(0, n_users))
            if not hist_by_user[u]:
                continue
            hist, pool, clicks = make_impression(u, rng)
            pos_ids = [pool[i] for i in np.where(clicks)[0]]
            neg_ids = [pool[i] for i in np.where(~clicks)[0]]
            if not pos_ids or len(neg_ids) < n_neg:
                continue
            pos = int(rng.choice(pos_ids))
            negs = list(map(int, rng.choice(neg_ids, size=n_neg, replace=False)))
            train.append((u, list(hist), pos, negs))
            edges.append((u, pos))  # clicked item joins the training graph

        # --- dev impressions (full slate with binary labels) ------------------
        dev = []
        tries = 0
        while len(dev) < n_dev and tries < n_dev * 20:
            tries += 1
            u = int(rng.integers(0, n_users))
            if not hist_by_user[u]:
                continue
            hist, pool, clicks = make_impression(u, rng)
            if clicks.sum() == 0 or clicks.all():
                continue  # AUC undefined
            dev.append((u, list(hist), list(map(int, pool)), list(map(int, clicks))))

        obj = cls(
            news_title=news_title, news_cat=news_cat, n_news=n_news, n_users=n_users,
            vocab_size=vocab_size, n_cat=int(news_cat.max()) + 1, title_len=title_len,
            max_hist=max_hist, llm_emb=llm_emb, train=train, dev=dev, _edges=edges,
        )
        return obj

    @classmethod
    def from_mind(
        cls,
        train_dir: str,
        dev_dir: str,
        title_len: int = 20,
        max_hist: int = 50,
        n_neg: int = 4,
        min_word_freq: int = 3,
        llm_dim: int = 64,
        llm_embeddings: dict[str, np.ndarray] | None = None,
        seed: int = 0,
    ) -> "MindData":
        """Parse real MIND ``news.tsv`` / ``behaviors.tsv``.

        ``llm_embeddings`` maps MIND news id -> vector (e.g. from
        ``precompute_llm_embeddings``). If omitted, a random-projection
        placeholder is used so the pipeline still runs end to end.
        """
        import os

        rng = np.random.default_rng(seed)

        def read_news(path):
            rows = {}
            with open(path, encoding="utf-8") as f:
                for line in f:
                    p = line.rstrip("\n").split("\t")
                    # id, category, subcategory, title, abstract, url, ...
                    rows[p[0]] = (p[1], p[3])
            return rows

        news_rows = read_news(os.path.join(train_dir, "news.tsv"))
        news_rows.update(read_news(os.path.join(dev_dir, "news.tsv")))

        # vocab + category maps
        from collections import Counter

        wf = Counter()
        for _, title in news_rows.values():
            wf.update(title.lower().split())
        vocab = {"<pad>": PAD}
        for w, c in wf.items():
            if c >= min_word_freq:
                vocab[w] = len(vocab)
        cats = {c for c, _ in news_rows.values()}
        cat_map = {c: i + 1 for i, c in enumerate(sorted(cats))}

        nid_map = {"<pad>": PAD}  # MIND news id -> global index
        for nid in news_rows:
            nid_map[nid] = len(nid_map)
        n_news = len(nid_map)
        news_title = np.zeros((n_news, title_len), np.int64)
        news_cat = np.zeros((n_news,), np.int64)
        for nid, (cat, title) in news_rows.items():
            gi = nid_map[nid]
            toks = [vocab.get(w, PAD) for w in title.lower().split()][:title_len]
            news_title[gi, : len(toks)] = toks
            news_cat[gi] = cat_map[cat]

        # llm embeddings aligned to global index
        if llm_embeddings:
            dim = len(next(iter(llm_embeddings.values())))
            llm_emb = np.zeros((n_news, dim), np.float32)
            for nid, gi in nid_map.items():
                if nid in llm_embeddings:
                    llm_emb[gi] = llm_embeddings[nid]
        else:  # placeholder — replace with real embeddings for a fair LLM run
            proj = rng.normal(size=(title_len, llm_dim))
            llm_emb = (news_title @ proj).astype(np.float32)
        llm_emb = (llm_emb / (np.linalg.norm(llm_emb, axis=1, keepdims=True) + 1e-8)).astype(np.float32)

        uid_map: dict[str, int] = {}

        def uidx(uid):
            if uid not in uid_map:
                uid_map[uid] = len(uid_map)
            return uid_map[uid]

        def read_behaviors(path):
            out = []
            with open(path, encoding="utf-8") as f:
                for line in f:
                    p = line.rstrip("\n").split("\t")
                    # impr_id, user, time, history, impressions
                    user, hist, imprs = p[1], p[3], p[4]
                    hist_ids = [nid_map[h] for h in hist.split() if h in nid_map]
                    cands, labs = [], []
                    for it in imprs.split():
                        nid, lab = it.split("-")
                        if nid in nid_map:
                            cands.append(nid_map[nid])
                            labs.append(int(lab))
                    out.append((uidx(user), hist_ids, cands, labs))
            return out

        train_beh = read_behaviors(os.path.join(train_dir, "behaviors.tsv"))
        dev = read_behaviors(os.path.join(dev_dir, "behaviors.tsv"))

        train, edges = [], []
        for user, hist, cands, labs in train_beh:
            edges += [(user, n) for n in hist]
            pos = [c for c, l in zip(cands, labs) if l == 1]
            neg = [c for c, l in zip(cands, labs) if l == 0]
            if not pos or len(neg) < 1:
                continue
            for p in pos:
                sampled = list(rng.choice(neg, size=min(n_neg, len(neg)),
                                          replace=len(neg) < n_neg))
                train.append((user, hist, p, sampled))
                edges.append((user, p))

        n_users = len(uid_map)
        return cls(
            news_title=news_title, news_cat=news_cat, n_news=n_news, n_users=n_users,
            vocab_size=len(vocab), n_cat=len(cat_map) + 1, title_len=title_len,
            max_hist=max_hist, llm_emb=llm_emb, train=train, dev=dev, _edges=edges,
        )


def _to_batch(*arrays) -> Batch:
    (hist_title, hist_cat, hist_idx, hist_mask, cand_title, cand_cat,
     cand_idx, cand_mask, user_idx, labels) = arrays
    t = torch.from_numpy
    return Batch(
        hist_title=t(hist_title), hist_cat=t(hist_cat), hist_idx=t(hist_idx),
        hist_mask=t(hist_mask), cand_title=t(cand_title), cand_cat=t(cand_cat),
        cand_idx=t(cand_idx), cand_mask=t(cand_mask), user_idx=t(user_idx),
        labels=t(labels),
    )


def iter_batches(rows, batch_size, collate, shuffle=False, seed=0):
    order = list(range(len(rows)))
    if shuffle:
        random.Random(seed).shuffle(order)
    for i in range(0, len(order), batch_size):
        chunk = [rows[j] for j in order[i : i + batch_size]]
        yield collate(chunk)


In [ ]:
%%writefile models.py
"""Model zoo for the benchmark, all behind one interface.

Every model implements ``score(batch) -> logits[B, C]`` where C is the number of
candidates (training: 1 positive at slot 0 + negatives; eval: the padded slate).
The training loss and the eval metrics therefore treat every model identically,
so differences in the results table come only from the architecture — not from
the plumbing.

Families
--------
- NRMS       content, multi-head self-attention (Wu et al. 2019)
- NAML       content, CNN + multi-view additive attention (Wu et al. 2019)
- Fastformer content, additive (linear-complexity) attention (Wu et al. 2021)
- LightGCN   PURE GRAPH: user/news id embeddings on the click bipartite graph,
             no text at all (He et al. 2020)
- LLMEnc     LLM-as-encoder: frozen precomputed news embeddings + light head
             (the ONCE/DIRE-style discriminative LLM pipeline)
- HybridOpt  our optimisation: Fastformer news encoder (cheap) + candidate-aware
             user attention (CAUM-style) + a fused collaborative term, so it
             carries both content and graph signal and degrades gracefully on
             cold news where pure LightGCN fails.
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from data import PAD, Batch


# --------------------------------------------------------------------- blocks
class AdditiveAttention(nn.Module):
    """Additive (Bahdanau) attention pooling over a sequence -> single vector."""

    def __init__(self, dim: int, hidden: int = 200):
        super().__init__()
        self.proj = nn.Linear(dim, hidden)
        self.query = nn.Linear(hidden, 1, bias=False)

    def forward(self, x, mask=None):  # x [..., N, D], mask [..., N] bool
        a = self.query(torch.tanh(self.proj(x))).squeeze(-1)  # [..., N]
        if mask is not None:
            a = a.masked_fill(~mask, -1e9)
        w = torch.softmax(a, dim=-1)
        return torch.einsum("...n,...nd->...d", w, x)


class FastformerBlock(nn.Module):
    """One Fastformer layer: additive attention with linear complexity."""

    def __init__(self, dim: int, heads: int, dropout: float):
        super().__init__()
        assert dim % heads == 0
        self.h, self.dh, self.dim = heads, dim // heads, dim
        self.Wq = nn.Linear(dim, dim)
        self.Wk = nn.Linear(dim, dim)
        self.Wv = nn.Linear(dim, dim)
        self.q_att = nn.Linear(self.dh, 1)
        self.k_att = nn.Linear(self.dh, 1)
        self.Wr = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def _split(self, x, B, N):
        return x.view(B, N, self.h, self.dh).transpose(1, 2)  # [B, h, N, dh]

    def forward(self, x, mask=None):  # x [B, N, dim], mask [B, N] bool
        B, N, _ = x.shape
        Q = self._split(self.Wq(x), B, N)
        K = self._split(self.Wk(x), B, N)
        V = self._split(self.Wv(x), B, N)
        m = None if mask is None else mask[:, None, :]  # [B,1,N]

        aq = (self.q_att(Q).squeeze(-1)) / (self.dh ** 0.5)  # [B,h,N]
        if m is not None:
            aq = aq.masked_fill(~m, -1e9)
        aq = torch.softmax(aq, dim=-1)
        global_q = torch.einsum("bhn,bhnd->bhd", aq, Q)      # [B,h,dh]

        p = K * global_q.unsqueeze(2)                         # key-query interaction
        ak = (self.k_att(p).squeeze(-1)) / (self.dh ** 0.5)
        if m is not None:
            ak = ak.masked_fill(~m, -1e9)
        ak = torch.softmax(ak, dim=-1)
        global_k = torch.einsum("bhn,bhnd->bhd", ak, p)       # [B,h,dh]

        u = V * global_k.unsqueeze(2)                         # [B,h,N,dh]
        r = self.Wr(u.transpose(1, 2).reshape(B, N, self.dim))
        return self.drop(r + self.Wq(x))                      # residual with query


# --------------------------------------------------------------------- models
class NRMS(nn.Module):
    def __init__(self, d, cfg):
        super().__init__()
        D, drop, heads = cfg.dim, cfg.dropout, cfg.heads
        self.word_emb = nn.Embedding(d.vocab_size, D, padding_idx=PAD)
        self.news_mha = nn.MultiheadAttention(D, heads, dropout=drop, batch_first=True)
        self.news_pool = AdditiveAttention(D)
        self.user_mha = nn.MultiheadAttention(D, heads, dropout=drop, batch_first=True)
        self.user_pool = AdditiveAttention(D)
        self.drop = nn.Dropout(drop)

    def _news(self, title):  # [X, L] -> [X, D]
        e = self.drop(self.word_emb(title))
        o, _ = self.news_mha(e, e, e)
        return self.news_pool(self.drop(o))

    def score(self, b: Batch):
        B, H, L = b.hist_title.shape
        C = b.cand_title.shape[1]
        hist = self._news(b.hist_title.reshape(B * H, L)).view(B, H, -1)
        cand = self._news(b.cand_title.reshape(B * C, L)).view(B, C, -1)
        # MultiheadAttention returns NaN for a fully-padded row (cold user on
        # real MIND); let such rows attend freely, the additive pool masks them.
        kpm = ~b.hist_mask
        kpm = kpm.masked_fill(kpm.all(dim=1, keepdim=True), False)
        o, _ = self.user_mha(hist, hist, hist, key_padding_mask=kpm)
        user = self.user_pool(o, b.hist_mask)                 # [B, D]
        return torch.einsum("bd,bcd->bc", user, cand)


class NAML(nn.Module):
    def __init__(self, d, cfg):
        super().__init__()
        D, drop = cfg.dim, cfg.dropout
        self.word_emb = nn.Embedding(d.vocab_size, D, padding_idx=PAD)
        self.cat_emb = nn.Embedding(d.n_cat, D, padding_idx=PAD)
        self.conv = nn.Conv1d(D, D, kernel_size=3, padding=1)
        self.title_pool = AdditiveAttention(D)
        self.cat_dense = nn.Linear(D, D)
        self.view_pool = AdditiveAttention(D)
        self.user_pool = AdditiveAttention(D)
        self.drop = nn.Dropout(drop)

    def _news(self, title, cat):  # [X, L], [X] -> [X, D]
        e = self.drop(self.word_emb(title)).transpose(1, 2)   # [X, D, L]
        c = torch.relu(self.conv(e)).transpose(1, 2)          # [X, L, D]
        title_vec = self.title_pool(self.drop(c))             # [X, D]
        cat_vec = torch.relu(self.cat_dense(self.cat_emb(cat)))
        views = torch.stack([title_vec, cat_vec], dim=1)      # [X, 2, D]
        return self.view_pool(views)

    def score(self, b: Batch):
        B, H, L = b.hist_title.shape
        C = b.cand_title.shape[1]
        hist = self._news(b.hist_title.reshape(B * H, L), b.hist_cat.reshape(B * H)).view(B, H, -1)
        cand = self._news(b.cand_title.reshape(B * C, L), b.cand_cat.reshape(B * C)).view(B, C, -1)
        user = self.user_pool(hist, b.hist_mask)
        return torch.einsum("bd,bcd->bc", user, cand)


class Fastformer(nn.Module):
    def __init__(self, d, cfg):
        super().__init__()
        D, drop, heads = cfg.dim, cfg.dropout, cfg.heads
        self.word_emb = nn.Embedding(d.vocab_size, D, padding_idx=PAD)
        self.news_ff = FastformerBlock(D, heads, drop)
        self.news_pool = AdditiveAttention(D)
        self.user_ff = FastformerBlock(D, heads, drop)
        self.user_pool = AdditiveAttention(D)
        self.drop = nn.Dropout(drop)

    def _news(self, title):
        e = self.drop(self.word_emb(title))
        return self.news_pool(self.news_ff(e))

    def score(self, b: Batch):
        B, H, L = b.hist_title.shape
        C = b.cand_title.shape[1]
        hist = self._news(b.hist_title.reshape(B * H, L)).view(B, H, -1)
        cand = self._news(b.cand_title.reshape(B * C, L)).view(B, C, -1)
        user = self.user_pool(self.user_ff(hist, b.hist_mask), b.hist_mask)
        return torch.einsum("bd,bcd->bc", user, cand)


class LightGCN(nn.Module):
    """Pure collaborative graph model — no text is ever read."""

    def __init__(self, d, cfg):
        super().__init__()
        D = cfg.dim
        self.n_users = d.n_users
        self.n_layers = cfg.gcn_layers
        self.user_emb = nn.Embedding(d.n_users, D)
        self.news_emb = nn.Embedding(d.n_news, D)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.news_emb.weight, std=0.1)
        self.register_buffer("adj", d.build_adjacency())

    def _propagate(self):
        x = torch.cat([self.user_emb.weight, self.news_emb.weight], dim=0)
        agg = x
        for _ in range(self.n_layers):
            x = torch.sparse.mm(self.adj, x)
            agg = agg + x
        agg = agg / (self.n_layers + 1)
        return agg[: self.n_users], agg[self.n_users:]

    def score(self, b: Batch):
        u_all, n_all = self._propagate()
        user = u_all[b.user_idx]                              # [B, D]
        cand = n_all[b.cand_idx]                              # [B, C, D]
        return torch.einsum("bd,bcd->bc", user, cand)


class LLMNewsEncoder(nn.Module):
    """Frozen pretrained news embeddings + a light trainable projection.

    ``data.llm_emb`` holds one vector per news id — on synthetic data a simulated
    embedding, on real MIND the output of a strong lightweight sentence embedder
    (BGE-M3, Jina v3, or any sentence-transformer; see ``llm_embed.py``). Only the
    small projection head trains online, so this is cheap to serve. Maps a news
    index tensor ``[...]`` to ``[..., D]``."""

    def __init__(self, d, cfg):
        super().__init__()
        emb = torch.from_numpy(d.llm_emb)
        self.register_buffer("emb", emb)
        self.head = nn.Sequential(
            nn.Linear(emb.shape[1], cfg.dim), nn.ReLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.dim, cfg.dim),
        )

    def forward(self, idx):
        return self.head(self.emb[idx])


class LLMEnc(nn.Module):
    """LLM-as-encoder base: frozen pretrained news vectors + additive user pool.
    Mirrors the production pipeline (news embeddings computed offline once, almost
    no online cost)."""

    def __init__(self, d, cfg):
        super().__init__()
        self.news_idx = LLMNewsEncoder(d, cfg)
        self.user_pool = AdditiveAttention(cfg.dim)

    def score(self, b: Batch):
        hist = self.news_idx(b.hist_idx)                      # [B, H, D]
        cand = self.news_idx(b.cand_idx)                      # [B, C, D]
        user = self.user_pool(hist, b.hist_mask)
        return torch.einsum("bd,bcd->bc", user, cand)


class HybridOpt(nn.Module):
    """Our optimisation: linear-attention content + candidate-aware user +
    fused collaborative signal.

    - Fastformer news encoder keeps the per-token cost linear (vs NRMS' O(L^2)).
    - Candidate-aware attention (CAUM idea) re-weights the reading history
      *per candidate*, which plain additive/self attention cannot do.
    - A learnable-weighted collaborative dot product adds the graph signal, and
      because the content path always works, cold news does not collapse the way
      it does for pure LightGCN.
    """

    def __init__(self, d, cfg):
        super().__init__()
        D, drop, heads = cfg.dim, cfg.dropout, cfg.heads
        self.word_emb = nn.Embedding(d.vocab_size, D, padding_idx=PAD)
        self.news_ff = FastformerBlock(D, heads, drop)
        self.news_pool = AdditiveAttention(D)
        self.cand_proj = nn.Linear(D, D, bias=False)          # bilinear attn key
        self.user_id = nn.Embedding(d.n_users, D)
        self.news_id = nn.Embedding(d.n_news, D)
        nn.init.normal_(self.user_id.weight, std=0.1)
        nn.init.normal_(self.news_id.weight, std=0.1)
        self.log_lambda = nn.Parameter(torch.zeros(()))       # fusion weight
        self.drop = nn.Dropout(drop)
        self.scale = D ** 0.5

    def _news(self, title):
        e = self.drop(self.word_emb(title))
        return self.news_pool(self.news_ff(e))

    def score(self, b: Batch):
        B, H, L = b.hist_title.shape
        C = b.cand_title.shape[1]
        hist = self._news(b.hist_title.reshape(B * H, L)).view(B, H, -1)   # [B,H,D]
        cand = self._news(b.cand_title.reshape(B * C, L)).view(B, C, -1)   # [B,C,D]
        # candidate-aware user representation: attend history w.r.t. each candidate
        att = torch.einsum("bcd,bhd->bch", self.cand_proj(cand), hist) / self.scale
        att = att.masked_fill(~b.hist_mask[:, None, :], -1e9)
        att = torch.softmax(att, dim=-1)
        user_ca = torch.einsum("bch,bhd->bcd", att, hist)     # [B, C, D]
        content = (user_ca * cand).sum(-1)                    # [B, C]
        collab = torch.einsum("bd,bcd->bc", self.user_id(b.user_idx),
                              self.news_id(b.cand_idx))
        return content + torch.exp(self.log_lambda) * collab


# ===================================================================== #
#  TOP base: CAUM (candidate-aware) — strongest reproducible MIND model  #
# ===================================================================== #
class SelfAttnNewsEncoder(nn.Module):
    """NRMS-style news encoder. Shared by every improvement variant below so
    that only the *user* encoder differs between them (clean ablation)."""

    def __init__(self, d, cfg):
        super().__init__()
        D, drop, heads = cfg.dim, cfg.dropout, cfg.heads
        self.word_emb = nn.Embedding(d.vocab_size, D, padding_idx=PAD)
        self.mha = nn.MultiheadAttention(D, heads, dropout=drop, batch_first=True)
        self.pool = AdditiveAttention(D)
        self.drop = nn.Dropout(drop)

    def forward(self, title):  # [X, L] -> [X, D]
        e = self.drop(self.word_emb(title))
        o, _ = self.mha(e, e, e)
        return self.pool(self.drop(o))


def _safe_kpm(mask):
    """Key-padding mask that never masks a whole row (avoids MHA NaN)."""
    kpm = ~mask
    return kpm.masked_fill(kpm.all(dim=1, keepdim=True), False)


class CAUM(nn.Module):
    """Candidate-Aware User Modeling (Qi et al. 2022) — among the strongest
    reproducible content models on MIND. History is contextualised by
    self-attention, then re-weighted *per candidate* via an MLP interaction."""

    def __init__(self, d, cfg):
        super().__init__()
        D, drop, heads = cfg.dim, cfg.dropout, cfg.heads
        self.news = SelfAttnNewsEncoder(d, cfg)
        self.hist_sa = nn.MultiheadAttention(D, heads, dropout=drop, batch_first=True)
        self.inter = nn.Sequential(nn.Linear(2 * D, D), nn.ReLU(), nn.Linear(D, 1))

    def score(self, b: Batch):
        B, H, L = b.hist_title.shape
        C = b.cand_title.shape[1]
        hist = self.news(b.hist_title.reshape(B * H, L)).view(B, H, -1)
        cand = self.news(b.cand_title.reshape(B * C, L)).view(B, C, -1)
        h, _ = self.hist_sa(hist, hist, hist, key_padding_mask=_safe_kpm(b.hist_mask))
        D = h.size(-1)
        hh = h.unsqueeze(1).expand(B, C, H, D)
        cc = cand.unsqueeze(2).expand(B, C, H, D)
        a = self.inter(torch.cat([hh, cc], dim=-1)).squeeze(-1)   # [B, C, H]
        a = a.masked_fill(~b.hist_mask[:, None, :], -1e9)
        a = torch.softmax(a, dim=-1)
        user_ca = torch.einsum("bch,bchd->bcd", a, hh)            # [B, C, D]
        return (user_ca * cand).sum(-1)


# ===================================================================== #
#  Improvement blocks (2024-2026 LLM attention, ported to the user side) #
# ===================================================================== #
class DiffAttention(nn.Module):
    """Differential attention (Ye et al., ICLR 2025): two softmax maps minus one
    another cancel common-mode noise — here, denoises the noisy click history."""

    def __init__(self, dim, heads, dropout=0.0, depth=1):
        super().__init__()
        self.h, self.dh = heads, dim // heads
        self.q = nn.Linear(dim, 2 * dim)
        self.k = nn.Linear(dim, 2 * dim)
        self.v = nn.Linear(dim, dim)
        self.o = nn.Linear(dim, dim)
        self.lambda_init = 0.8 - 0.6 * math.exp(-0.3 * (depth - 1))
        self.lq1 = nn.Parameter(torch.randn(self.dh) * 0.1)
        self.lk1 = nn.Parameter(torch.randn(self.dh) * 0.1)
        self.lq2 = nn.Parameter(torch.randn(self.dh) * 0.1)
        self.lk2 = nn.Parameter(torch.randn(self.dh) * 0.1)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):  # x [B, N, dim]
        B, N, _ = x.shape
        q = self.q(x).view(B, N, 2, self.h, self.dh)
        k = self.k(x).view(B, N, 2, self.h, self.dh)
        v = self.v(x).view(B, N, self.h, self.dh)

        def attn(qi, ki):
            s = torch.einsum("bnhd,bmhd->bhnm", qi, ki) / (self.dh ** 0.5)
            if mask is not None:
                s = s.masked_fill(~mask[:, None, None, :], -1e9)
            return torch.softmax(s, dim=-1)

        a1 = attn(q[:, :, 0], k[:, :, 0])
        a2 = attn(q[:, :, 1], k[:, :, 1])
        lam = (torch.exp((self.lq1 * self.lk1).sum())
               - torch.exp((self.lq2 * self.lk2).sum()) + self.lambda_init)
        a = self.drop(a1 - lam * a2)
        out = torch.einsum("bhnm,bmhd->bnhd", a, v).reshape(B, N, -1)
        out = self.norm(out) * (1 - self.lambda_init)
        return self.o(out)


class MLASelfAttn(nn.Module):
    """Multi-head Latent Attention (DeepSeek-V2/V3): K and V come from a low-rank
    latent projection of the history — the per-position cache is the small
    latent, not full K/V."""

    def __init__(self, dim, heads, latent, dropout=0.0):
        super().__init__()
        self.h, self.dh = heads, dim // heads
        self.q = nn.Linear(dim, dim)
        self.kv_down = nn.Linear(dim, latent)     # compress (the cached state)
        self.k_up = nn.Linear(latent, dim)
        self.v_up = nn.Linear(latent, dim)
        self.o = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def _split(self, t, B, N):
        return t.view(B, N, self.h, self.dh).transpose(1, 2)

    def forward(self, x, mask=None):
        B, N, _ = x.shape
        c = self.kv_down(x)
        q = self._split(self.q(x), B, N)
        k = self._split(self.k_up(c), B, N)
        v = self._split(self.v_up(c), B, N)
        s = torch.einsum("bhnd,bhmd->bhnm", q, k) / (self.dh ** 0.5)
        if mask is not None:
            s = s.masked_fill(~mask[:, None, None, :], -1e9)
        a = self.drop(torch.softmax(s, dim=-1))
        o = torch.einsum("bhnm,bhmd->bhnd", a, v).transpose(1, 2).reshape(B, N, -1)
        return self.o(o)


class SSMBlock(nn.Module):
    """Lightweight selective diagonal state-space layer (Mamba-style): an O(N)
    input-dependent linear recurrence over the history. Not the full Mamba CUDA
    kernel — a compact SSM that captures the linear-time long-range idea."""

    def __init__(self, dim):
        super().__init__()
        self.a = nn.Linear(dim, dim)
        self.b = nn.Linear(dim, dim)
        self.c = nn.Linear(dim, dim)
        self.gate = nn.Linear(dim, dim)

    def forward(self, x, mask=None):  # x [B, N, D]
        B, N, D = x.shape
        decay = torch.sigmoid(self.a(x))     # selective (input-dependent) forget
        bx = self.b(x) * x
        cc = self.c(x)
        h = x.new_zeros(B, D)
        ys = []
        for t in range(N):
            new_h = decay[:, t] * h + bx[:, t]
            if mask is not None:
                h = torch.where(mask[:, t].unsqueeze(-1), new_h, h)  # freeze on pad
            else:
                h = new_h
            ys.append(cc[:, t] * h)
        y = torch.stack(ys, dim=1)
        return y * torch.sigmoid(self.gate(x))


# ------ RoPE helpers + RoPE news encoder ------
def _rope_tables(seq, dh, device, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dh, 2, device=device).float() / dh))
    t = torch.arange(seq, device=device).float()[:, None] * inv[None, :]
    return torch.cos(t), torch.sin(t)                 # [seq, dh/2]


def _apply_rope(x, cos, sin):  # x [., h, N, dh]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos, sin = cos[None, None], sin[None, None]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class RopeNewsEncoder(nn.Module):
    """News encoder whose token self-attention uses rotary positions (NRMS has
    no positional signal at all)."""

    def __init__(self, d, cfg):
        super().__init__()
        D, drop, heads = cfg.dim, cfg.dropout, cfg.heads
        self.h, self.dh, self.D = heads, D // heads, D
        self.word_emb = nn.Embedding(d.vocab_size, D, padding_idx=PAD)
        self.q = nn.Linear(D, D)
        self.k = nn.Linear(D, D)
        self.v = nn.Linear(D, D)
        self.o = nn.Linear(D, D)
        self.pool = AdditiveAttention(D)
        self.drop = nn.Dropout(drop)

    def forward(self, title):  # [X, L]
        X, L = title.shape
        e = self.drop(self.word_emb(title))

        def sp(t):
            return t.view(X, L, self.h, self.dh).transpose(1, 2)

        q, k, v = sp(self.q(e)), sp(self.k(e)), sp(self.v(e))
        cos, sin = _rope_tables(L, self.dh, title.device)
        q, k = _apply_rope(q, cos, sin), _apply_rope(k, cos, sin)
        s = torch.einsum("xhld,xhmd->xhlm", q, k) / (self.dh ** 0.5)
        a = torch.softmax(s, dim=-1)
        o = torch.einsum("xhlm,xhmd->xhld", a, v).transpose(1, 2).reshape(X, L, self.D)
        return self.pool(self.drop(self.o(o)))


# ===================================================================== #
#  Improvement models: shared NRMS news encoder, different user encoder  #
# ===================================================================== #
class _ContentTwoTower(nn.Module):
    """Encode history + candidates, then delegate user modeling to subclasses.

    ``build_news=False`` skips the learned word-level news encoder (used when a
    subclass plugs in frozen pretrained news embeddings instead)."""

    def __init__(self, d, cfg, news=None, build_news=True):
        super().__init__()
        self.pretrained = False
        if build_news:
            self.news = news or SelfAttnNewsEncoder(d, cfg)

    def _encode(self, b: Batch):
        if self.pretrained:                                   # frozen BGE/Jina emb
            return self.news_idx(b.hist_idx), self.news_idx(b.cand_idx)
        B, H, L = b.hist_title.shape
        C = b.cand_title.shape[1]
        hist = self.news(b.hist_title.reshape(B * H, L)).view(B, H, -1)
        cand = self.news(b.cand_title.reshape(B * C, L)).view(B, C, -1)
        return hist, cand


class NrmsDiff(_ContentTwoTower):
    def __init__(self, d, cfg):
        super().__init__(d, cfg)
        self.user = DiffAttention(cfg.dim, cfg.heads, cfg.dropout)
        self.pool = AdditiveAttention(cfg.dim)

    def score(self, b: Batch):
        hist, cand = self._encode(b)
        user = self.pool(self.user(hist, b.hist_mask), b.hist_mask)
        return torch.einsum("bd,bcd->bc", user, cand)


class NrmsMLA(_ContentTwoTower):
    def __init__(self, d, cfg):
        super().__init__(d, cfg)
        self.user = MLASelfAttn(cfg.dim, cfg.heads, cfg.latent, cfg.dropout)
        self.pool = AdditiveAttention(cfg.dim)

    def score(self, b: Batch):
        hist, cand = self._encode(b)
        user = self.pool(self.user(hist, b.hist_mask), b.hist_mask)
        return torch.einsum("bd,bcd->bc", user, cand)


class NrmsSSM(_ContentTwoTower):
    def __init__(self, d, cfg):
        super().__init__(d, cfg)
        self.user = SSMBlock(cfg.dim)
        self.pool = AdditiveAttention(cfg.dim)

    def score(self, b: Batch):
        hist, cand = self._encode(b)
        user = self.pool(self.user(hist, b.hist_mask), b.hist_mask)
        return torch.einsum("bd,bcd->bc", user, cand)


class NrmsRoPE(_ContentTwoTower):
    def __init__(self, d, cfg):
        super().__init__(d, cfg, news=RopeNewsEncoder(d, cfg))
        self.user_mha = nn.MultiheadAttention(cfg.dim, cfg.heads,
                                              dropout=cfg.dropout, batch_first=True)
        self.pool = AdditiveAttention(cfg.dim)
        self.recency = nn.Parameter(torch.zeros(d.max_hist))  # learned recency bias

    def score(self, b: Batch):
        hist, cand = self._encode(b)
        o, _ = self.user_mha(hist, hist, hist, key_padding_mask=_safe_kpm(b.hist_mask))
        # additive attention pooling with an added ALiBi-style recency bias
        w = self.pool.query(torch.tanh(self.pool.proj(o))).squeeze(-1)   # [B, H]
        w = w + self.recency[: o.size(1)][None, :]
        w = w.masked_fill(~b.hist_mask, -1e9)
        w = torch.softmax(w, dim=-1)
        user = torch.einsum("bn,bnd->bd", w, o)
        return torch.einsum("bd,bcd->bc", user, cand)


class NrmsMulti(_ContentTwoTower):
    """Multi-interest user modeling (MINS/MINER-style poly-attention): several
    interest vectors, candidate scored by its best-matching interest."""

    def __init__(self, d, cfg):
        super().__init__(d, cfg)
        self.W1 = nn.Linear(cfg.dim, cfg.dim)
        self.W2 = nn.Linear(cfg.dim, cfg.n_interest, bias=False)

    def score(self, b: Batch):
        hist, cand = self._encode(b)
        a = self.W2(torch.tanh(self.W1(hist)))              # [B, H, K]
        a = a.masked_fill(~b.hist_mask[:, :, None], -1e9)
        a = torch.softmax(a, dim=1)                          # over history
        user_k = torch.einsum("bhk,bhd->bkd", a, hist)      # [B, K, D]
        s = torch.einsum("bkd,bcd->bkc", user_k, cand)      # [B, K, C]
        return s.max(dim=1).values                           # best interest per cand


# ===================================================================== #
#  Graph fusion + contrastive learning + the combined "super" model      #
# ===================================================================== #
class GraphProp(nn.Module):
    """LightGCN propagation → collaborative user/news embeddings (reused by the
    graph-enhanced and super models)."""

    def __init__(self, d, cfg):
        super().__init__()
        D = cfg.dim
        self.n_users = d.n_users
        self.n_layers = cfg.gcn_layers
        self.user_emb = nn.Embedding(d.n_users, D)
        self.news_emb = nn.Embedding(d.n_news, D)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.news_emb.weight, std=0.1)
        self.register_buffer("adj", d.build_adjacency())

    def forward(self):
        x = torch.cat([self.user_emb.weight, self.news_emb.weight], dim=0)
        agg = x
        for _ in range(self.n_layers):
            x = torch.sparse.mm(self.adj, x)
            agg = agg + x
        agg = agg / (self.n_layers + 1)
        return agg[: self.n_users], agg[self.n_users:]


class GraphRec(_ContentTwoTower):
    """Graph direction: NRMS content two-tower fused with LightGCN-propagated
    collaborative embeddings (real GNN message passing, unlike hybridopt's plain
    id embeddings). Score = content + learnable-weight * collaborative."""

    def __init__(self, d, cfg):
        super().__init__(d, cfg)
        self.user_pool = AdditiveAttention(cfg.dim)
        self.graph = GraphProp(d, cfg)
        self.gate = nn.Parameter(torch.zeros(()))

    def score(self, b: Batch):
        hist, cand = self._encode(b)
        cu = self.user_pool(hist, b.hist_mask)                 # content user
        content = torch.einsum("bd,bcd->bc", cu, cand)
        gu_all, gn_all = self.graph()
        collab = torch.einsum("bd,bcd->bc", gu_all[b.user_idx], gn_all[b.cand_idx])
        return content + torch.exp(self.gate) * collab


class NrmsCL(_ContentTwoTower):
    """Contrastive-learning direction: plain NRMS two-tower trained with an extra
    self-supervised InfoNCE loss (CL4SRec / SimCSE-style) — two dropout views of
    each user are pulled together, other users in the batch pushed apart."""

    def __init__(self, d, cfg):
        super().__init__(d, cfg)
        self.user_pool = AdditiveAttention(cfg.dim)
        self.drop = nn.Dropout(cfg.dropout)
        self.tau = cfg.cl_tau
        self.cl_weight = cfg.cl_weight

    def score(self, b: Batch):
        hist, cand = self._encode(b)
        user = self.user_pool(hist, b.hist_mask)
        return torch.einsum("bd,bcd->bc", user, cand)

    def extra_loss(self, b: Batch):
        hist, _ = self._encode(b)
        v1 = F.normalize(self.user_pool(self.drop(hist), b.hist_mask), dim=-1)
        v2 = F.normalize(self.user_pool(self.drop(hist), b.hist_mask), dim=-1)
        logits = (v1 @ v2.t()) / self.tau                      # [B, B]
        labels = torch.arange(v1.size(0), device=v1.device)
        return self.cl_weight * F.cross_entropy(logits, labels)


class SuperRec(_ContentTwoTower):
    """The 'combine everything' model, to stack against llmenc:
      • self-attention news encoder (NRMS)
      • DIFFERENTIAL attention to denoise the click history
      • CANDIDATE-AWARE attention over the denoised history (CAUM idea)
      • GRAPH (LightGCN) collaborative fusion
      • CONTRASTIVE self-supervised auxiliary loss
    """

    def __init__(self, d, cfg):
        pretrained = getattr(cfg, "news_encoder", "learned") == "pretrained"
        super().__init__(d, cfg, build_news=not pretrained)
        self.pretrained = pretrained
        if pretrained:                                        # BGE-M3 / Jina news emb
            self.news_idx = LLMNewsEncoder(d, cfg)
        self.hist_diff = DiffAttention(cfg.dim, cfg.heads, cfg.dropout)
        self.cand_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.graph = GraphProp(d, cfg)
        self.gate = nn.Parameter(torch.zeros(()))
        self.user_pool = AdditiveAttention(cfg.dim)
        self.drop = nn.Dropout(cfg.dropout)
        self.tau, self.cl_weight = cfg.cl_tau, cfg.cl_weight
        self.scale = cfg.dim ** 0.5

    def score(self, b: Batch):
        hist, cand = self._encode(b)
        h = self.hist_diff(hist, b.hist_mask)                  # denoised history
        att = torch.einsum("bcd,bhd->bch", self.cand_proj(cand), h) / self.scale
        att = att.masked_fill(~b.hist_mask[:, None, :], -1e9)
        att = torch.softmax(att, dim=-1)
        user_ca = torch.einsum("bch,bhd->bcd", att, h)         # candidate-aware user
        content = (user_ca * cand).sum(-1)
        gu_all, gn_all = self.graph()
        collab = torch.einsum("bd,bcd->bc", gu_all[b.user_idx], gn_all[b.cand_idx])
        return content + torch.exp(self.gate) * collab

    def extra_loss(self, b: Batch):
        hist, _ = self._encode(b)
        v1 = F.normalize(self.user_pool(self.drop(hist), b.hist_mask), dim=-1)
        v2 = F.normalize(self.user_pool(self.drop(hist), b.hist_mask), dim=-1)
        logits = (v1 @ v2.t()) / self.tau
        labels = torch.arange(v1.size(0), device=v1.device)
        return self.cl_weight * F.cross_entropy(logits, labels)


REGISTRY = {
    # --- top base models (reference) ---
    "nrms": NRMS,
    "naml": NAML,
    "fastformer": Fastformer,
    "caum": CAUM,               # strongest reproducible MIND content model
    "lightgcn": LightGCN,       # pure graph
    "llmenc": LLMEnc,           # LLM-as-encoder
    # --- improvement variants (2024-2026 attention, ablation on user encoder) ---
    "nrms_diff": NrmsDiff,      # differential attention (denoise)
    "nrms_mla": NrmsMLA,        # multi-head latent attention (low-rank KV)
    "nrms_ssm": NrmsSSM,        # Mamba-style selective SSM (linear, long history)
    "nrms_rope": NrmsRoPE,      # rotary positions + recency bias
    "nrms_multi": NrmsMulti,    # multi-interest poly-attention
    "graphrec": GraphRec,       # content + LightGCN graph fusion (GNN message passing)
    "nrms_cl": NrmsCL,          # contrastive learning (InfoNCE self-supervised aux)
    "hybridopt": HybridOpt,     # Fastformer + candidate-aware + collaborative fusion
    "supermodel": SuperRec,     # diff-attn + candidate-aware + graph + contrastive (all-in-one)
}

BASE_MODELS = ["nrms", "naml", "fastformer", "caum", "lightgcn", "llmenc"]
IMPROVED_MODELS = ["nrms_diff", "nrms_mla", "nrms_ssm", "nrms_rope", "nrms_multi",
                   "graphrec", "nrms_cl", "hybridopt", "supermodel"]
# Lean, non-diluted default: 3 strong transformers + graph + LLM-encoder + the
# consolidated super model. The rest stay available as an ablation menu.
CORE_MODELS = ["nrms", "fastformer", "caum", "lightgcn", "llmenc", "supermodel"]


def build_model(name: str, data, cfg) -> nn.Module:
    if name not in REGISTRY:
        raise KeyError(f"unknown model '{name}'; choose from {list(REGISTRY)}")
    return REGISTRY[name](data, cfg)


In [ ]:
%%writefile bench.py
"""Unified train / eval / timing harness for the news-rec benchmark.

Run examples
------------
# synthetic controlled demo (default) — every model, small + fast:
python bench.py --source synthetic --epochs 3

# a single model:
python bench.py --models nrms hybridopt --epochs 5

# real MIND (download MINDsmall from https://msnews.github.io first):
python bench.py --source mind \
    --mind-train /data/MINDsmall_train --mind-dev /data/MINDsmall_dev --epochs 4

Because the loss (softmax cross-entropy over candidates) and the metrics
(AUC / MRR / nDCG) are shared, the only thing that varies between rows of the
results table is the model architecture and its speed.
"""
from __future__ import annotations

import argparse
import json
import os
import time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F

import metrics
from data import MindData, iter_batches
from models import BASE_MODELS, CORE_MODELS, IMPROVED_MODELS, REGISTRY, build_model


@dataclass
class Cfg:
    dim: int = 64
    dropout: float = 0.2
    heads: int = 2
    gcn_layers: int = 2
    news_encoder: str = "learned"   # "learned" | "pretrained" (BGE/Jina) — supermodel
    latent: int = 16          # MLA latent dim (nrms_mla)
    n_interest: int = 4       # number of interests (nrms_multi)
    cl_tau: float = 0.1       # contrastive temperature (nrms_cl, supermodel)
    cl_weight: float = 0.1    # contrastive aux-loss weight
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 3
    batch_size: int = 64
    eval_batch_size: int = 128


def train_model(model, data, cfg, device):
    model.to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    t0 = time.perf_counter()
    per_epoch = []
    for _ in range(cfg.epochs):
        e0 = time.perf_counter()
        total, nb = 0.0, 0
        for batch in iter_batches(data.train, cfg.batch_size, data.collate_train,
                                  shuffle=True):
            batch = batch.to(device)
            logits = model.score(batch)                       # [B, C], pos at 0
            target = torch.zeros(logits.size(0), dtype=torch.long, device=device)
            loss = F.cross_entropy(logits, target)
            if hasattr(model, "extra_loss"):                  # e.g. contrastive aux
                loss = loss + model.extra_loss(batch)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
            nb += 1
        per_epoch.append(time.perf_counter() - e0)
    return time.perf_counter() - t0, float(np.mean(per_epoch)), total / max(nb, 1)


@torch.no_grad()
def evaluate(model, data, cfg, device):
    model.eval()
    impressions = []
    t0 = time.perf_counter()
    for batch in iter_batches(data.dev, cfg.eval_batch_size, data.collate_eval):
        batch = batch.to(device)
        logits = model.score(batch)
        logits = logits.masked_fill(~batch.cand_mask, float("-inf"))
        scores = logits.cpu().numpy()
        labels = batch.labels.cpu().numpy()
        mask = batch.cand_mask.cpu().numpy()
        for i in range(scores.shape[0]):
            m = mask[i]
            impressions.append((labels[i][m], scores[i][m]))
    eval_time = time.perf_counter() - t0
    result = metrics.aggregate(impressions)
    result["dev_impr_per_s"] = len(impressions) / max(eval_time, 1e-9)
    return result


def n_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def format_table(rows: list[dict]) -> str:
    cols = ["model", "auc", "mrr", "ndcg@5", "ndcg@10",
            "params", "train_s", "s/epoch", "infer_impr/s"]
    head = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    lines = [head, sep]
    for r in rows:
        lines.append("| " + " | ".join([
            r["model"],
            f"{r['auc']:.4f}", f"{r['mrr']:.4f}",
            f"{r['ndcg@5']:.4f}", f"{r['ndcg@10']:.4f}",
            f"{r['params']:,}", f"{r['train_s']:.1f}",
            f"{r['s_per_epoch']:.2f}", f"{r['infer_impr_per_s']:.0f}",
        ]) + " |")
    return "\n".join(lines)


def main():
    ap = argparse.ArgumentParser(description="News recommendation benchmark")
    ap.add_argument("--source", choices=["synthetic", "mind"], default="synthetic")
    ap.add_argument("--mind-train", default=None)
    ap.add_argument("--mind-dev", default=None)
    ap.add_argument("--news-emb", default=None,
                    help="path to news_emb.npz (from llm_embed.py) for real LLM/BGE "
                         "embeddings; wired into from_mind so llmenc/supermodel are strong")
    ap.add_argument("--models", nargs="+", default=["core"],
                    help="model names, or the shortcuts: core / all / base / improved")
    ap.add_argument("--news", choices=["learned", "pretrained"], default="learned",
                    help="supermodel news encoder: learned words, or frozen BGE/Jina embeddings")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--dim", type=int, default=64)
    ap.add_argument("--heads", type=int, default=2)
    ap.add_argument("--gcn-layers", type=int, default=2)
    ap.add_argument("--latent", type=int, default=16)
    ap.add_argument("--n-interest", type=int, default=4)
    ap.add_argument("--cl-tau", type=float, default=0.1)
    ap.add_argument("--cl-weight", type=float, default=0.1)
    ap.add_argument("--dropout", type=float, default=0.2)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--batch-size", type=int, default=64)
    ap.add_argument("--eval-batch", type=int, default=128)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--threads", type=int, default=0, help="torch CPU threads (0=default)")
    ap.add_argument("--out", default="results")
    # synthetic knobs
    ap.add_argument("--syn-users", type=int, default=2000)
    ap.add_argument("--syn-news", type=int, default=3000)
    ap.add_argument("--syn-train", type=int, default=8000)
    ap.add_argument("--syn-dev", type=int, default=2000)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if args.threads:
        torch.set_num_threads(args.threads)
    device = ("cuda" if torch.cuda.is_available() else "cpu") if args.device == "auto" else args.device

    if args.source == "mind":
        assert args.mind_train and args.mind_dev, "provide --mind-train and --mind-dev"
        print(f"Loading real MIND from {args.mind_train} / {args.mind_dev} ...")
        emb = None
        if args.news_emb:
            z = np.load(args.news_emb, allow_pickle=True)
            emb = {str(i): v for i, v in zip(z["ids"], z["vecs"])}
            print(f"  loaded {len(emb)} precomputed news embeddings from {args.news_emb}")
        data = MindData.from_mind(args.mind_train, args.mind_dev,
                                  llm_embeddings=emb, seed=args.seed)
    else:
        print("Generating synthetic MIND-shaped dataset (controlled demo) ...")
        data = MindData.synthetic(
            n_users=args.syn_users, n_news=args.syn_news,
            n_train=args.syn_train, n_dev=args.syn_dev, seed=args.seed,
        )
    print(f"  users={data.n_users}  news={data.n_news}  vocab={data.vocab_size}  "
          f"cats={data.n_cat}  train={len(data.train)}  dev={len(data.dev)}  device={device}")

    cfg = Cfg(dim=args.dim, dropout=args.dropout, heads=args.heads,
              gcn_layers=args.gcn_layers, news_encoder=args.news,
              latent=args.latent, n_interest=args.n_interest,
              cl_tau=args.cl_tau, cl_weight=args.cl_weight,
              lr=args.lr, epochs=args.epochs, batch_size=args.batch_size,
              eval_batch_size=args.eval_batch)

    # expand shortcuts: core / all / base / improved
    expand = {"core": CORE_MODELS, "all": list(REGISTRY),
              "base": BASE_MODELS, "improved": IMPROVED_MODELS}
    model_names = []
    for m in args.models:
        model_names.extend(expand.get(m, [m]))

    rows = []
    for name in model_names:
        torch.manual_seed(args.seed)  # same init budget per model
        model = build_model(name, data, cfg)
        train_s, s_epoch, last_loss = train_model(model, data, cfg, device)
        res = evaluate(model, data, cfg, device)
        row = {
            "model": name, **{k: res[k] for k in ("auc", "mrr", "ndcg@5", "ndcg@10")},
            "params": n_params(model), "train_s": train_s, "s_per_epoch": s_epoch,
            "infer_impr_per_s": res["dev_impr_per_s"], "final_loss": last_loss,
        }
        rows.append(row)
        print(f"[{name:10s}] AUC={row['auc']:.4f} MRR={row['mrr']:.4f} "
              f"nDCG@10={row['ndcg@10']:.4f} | {row['params']:,} params | "
              f"{train_s:.1f}s train | {row['infer_impr_per_s']:.0f} impr/s")

    rows.sort(key=lambda r: r["auc"], reverse=True)
    table = format_table(rows)
    print("\n" + table)

    os.makedirs(args.out, exist_ok=True)
    with open(os.path.join(args.out, "results.json"), "w") as f:
        json.dump({"source": args.source, "config": vars(args), "rows": rows}, f, indent=2)
    with open(os.path.join(args.out, "results.md"), "w") as f:
        f.write(f"# Benchmark results ({args.source})\n\n{table}\n")
    print(f"\nSaved -> {args.out}/results.json, {args.out}/results.md")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile llm_embed.py
"""Precompute strong lightweight news embeddings (BGE-M3 / Jina / ST) for MIND.

The ``llmenc`` base model and the ``supermodel`` (with ``--news pretrained``) read
``MindData.llm_emb``. On synthetic data that is a simulated embedding; for a real
run on MIND, precompute embeddings of the news titles (or title + abstract) with a
modern sentence embedder and pass the dict to ``MindData.from_mind(..., llm_embeddings=)``.

    pip install sentence-transformers
    # lightweight & strong, pick one:
    python llm_embed.py --news /data/MINDsmall_train/news.tsv --out news_emb.npz \
        --model BAAI/bge-small-en-v1.5            # ~33M, very light (default)
    #   --model BAAI/bge-m3                       # multilingual, stronger (~560M)
    #   --model jinaai/jina-embeddings-v3         # multilingual (needs trust-remote-code)
    #   --model jinaai/jina-embeddings-v2-small-en  # ~33M, very light

Then in your own script:

    import numpy as np
    from data import MindData
    z = np.load("news_emb.npz", allow_pickle=True)
    emb = {nid: v for nid, v in zip(z["ids"], z["vecs"])}
    data = MindData.from_mind(train_dir, dev_dir, llm_embeddings=emb)

BGE-M3 and Jina are good news-embedding choices: multilingual, instruction-free,
and small enough to run on CPU. The projection head in the model adapts whatever
dimension the encoder outputs, so any sentence encoder drops in.
"""
from __future__ import annotations

import argparse

import numpy as np


def read_titles(path: str) -> tuple[list[str], list[str]]:
    ids, texts = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            ids.append(p[0])
            title, abstract = p[3], (p[4] if len(p) > 4 else "")
            texts.append((title + ". " + abstract).strip())
    return ids, texts


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--news", required=True, help="path to MIND news.tsv")
    ap.add_argument("--out", default="news_emb.npz")
    ap.add_argument("--model", default="BAAI/bge-small-en-v1.5",
                    help="e.g. BAAI/bge-m3, jinaai/jina-embeddings-v3, or any ST model")
    ap.add_argument("--batch-size", type=int, default=256)
    ap.add_argument("--trust-remote-code", action="store_true",
                    help="required for jina-embeddings-v3 and some custom encoders")
    args = ap.parse_args()

    from sentence_transformers import SentenceTransformer

    ids, texts = read_titles(args.news)
    enc = SentenceTransformer(args.model, trust_remote_code=args.trust_remote_code)
    vecs = enc.encode(texts, batch_size=args.batch_size, show_progress_bar=True,
                      normalize_embeddings=True)
    np.savez(args.out, ids=np.array(ids), vecs=np.asarray(vecs, np.float32))
    print(f"Saved {len(ids)} embeddings of dim {vecs.shape[1]} -> {args.out}")


if __name__ == "__main__":
    main()


### 2. Cấu hình
Để `MIND_TRAIN=None` → chạy **synthetic demo** (không cần dữ liệu). Điền đường dẫn MIND để chạy thật. Ví dụ đường dẫn Kaggle điển hình:
`/kaggle/input/mind-news-dataset/MINDsmall_train`.

In [ ]:
# === CONFIG ===
DOWNLOAD_MIND = True           # tự tải MIND từ internet (cần Settings -> Internet = On)
MIND_SIZE     = 'small'        # 'small' (~80MB, khuyên dùng) hoặc 'large'
MIND_TRAIN = None   # để None -> tự tải/tự dò; hoặc trỏ tay '/kaggle/input/.../MINDsmall_train'
MIND_DEV   = None
MODELS     = ['core']          # 'core' | 'base' | 'improved' | 'all' | vd ['nrms','llmenc','supermodel']
EPOCHS     = 4
DIM        = 64
EVAL_BATCH = 32                # giảm nếu CAUM OOM trên impression lớn của MIND
USE_BGE    = False             # True + Internet On + có MIND -> tính embedding BGE thật cho llmenc/supermodel
BGE_MODEL  = 'BAAI/bge-small-en-v1.5'
NEWS_EMB   = None

### 2a. Tự tải MIND từ internet (cần **Settings → Internet = On**)
MIND-small ~80MB, tải ~1–2 phút. Dùng URL chính thức của Microsoft. Nếu Internet Off hoặc tải lỗi → notebook sẽ tự chạy synthetic (hoặc bạn *Add Input* dataset MIND).

In [ ]:
import os, urllib.request, zipfile
BASES = {'small': 'https://mind201910small.blob.core.windows.net/release',
         'large': 'https://mind201910large.blob.core.windows.net/release'}
tag = 'MINDsmall' if MIND_SIZE == 'small' else 'MINDlarge'
if DOWNLOAD_MIND and not (MIND_TRAIN and os.path.exists(os.path.join(MIND_TRAIN, 'news.tsv'))):
    dst = '/kaggle/working/MIND' if os.path.isdir('/kaggle/working') else './MIND'
    os.makedirs(dst, exist_ok=True)
    try:
        for split in ['train', 'dev']:
            z = f'{dst}/{tag}_{split}.zip'
            if not os.path.exists(z):
                url = f'{BASES[MIND_SIZE]}/{tag}_{split}.zip'
                print('Đang tải', url, '...')
                urllib.request.urlretrieve(url, z)
            outd = f'{dst}/{tag}_{split}'
            if not os.path.exists(os.path.join(outd, 'news.tsv')):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(outd)
        MIND_TRAIN, MIND_DEV = f'{dst}/{tag}_train', f'{dst}/{tag}_dev'
        print('MIND sẵn sàng:', MIND_TRAIN, '|', MIND_DEV)
    except Exception as e:
        print('Tải MIND lỗi:', repr(e))
        print('-> Bật Settings > Internet = On, hoặc Add Input > Datasets dataset MIND. '
              'Tạm thời notebook sẽ chạy synthetic.')
else:
    print('Bỏ qua tải (DOWNLOAD_MIND=False hoặc MIND_TRAIN đã có).')

### 2b. Tự dò đường dẫn MIND (chạy sau khi *Add Input → Datasets* dataset MIND)
Ô này quét `/kaggle/input` tìm `news.tsv` và **tự đặt** `MIND_TRAIN`/`MIND_DEV`. Nếu chưa gắn dataset, nó báo trống và notebook sẽ chạy synthetic.

In [ ]:
import glob, os
hits = glob.glob('/kaggle/input/**/news.tsv', recursive=True)
dirs = sorted({os.path.dirname(h) for h in hits})
print('Thư mục chứa news.tsv:', dirs if dirs else '(chưa gắn dataset MIND)')
if not MIND_TRAIN and dirs:
    tr = [d for d in dirs if 'train' in d.lower()]
    dv = [d for d in dirs if ('dev' in d.lower() or 'valid' in d.lower())]
    # ưu tiên MINDsmall nếu có cả small/large
    tr = [d for d in tr if 'small' in d.lower()] or tr
    dv = [d for d in dv if 'small' in d.lower()] or dv
    if tr and dv:
        MIND_TRAIN, MIND_DEV = tr[0], dv[0]
        print('Tự đặt MIND_TRAIN =', MIND_TRAIN)
        print('Tự đặt MIND_DEV   =', MIND_DEV)
    else:
        print('Không tự nhận ra train/dev. Hãy gán tay MIND_TRAIN/MIND_DEV theo danh sách trên.')
else:
    print('MIND_TRAIN hiện =', MIND_TRAIN)

### 2c. ✅ Xác nhận đã có MIND THẬT (đọc để biết chắc không phải synthetic)
In số dòng `news.tsv` / `behaviors.tsv`. MIND-small chuẩn: **news ~51k, train behaviors ~156k, dev ~73k**. Thấy đúng cỡ này là data thật.

In [ ]:
import os
def _count(p):
    try:
        with open(p, encoding='utf-8', errors='ignore') as f:
            return sum(1 for _ in f)
    except Exception:
        return -1
print('=' * 60)
if MIND_TRAIN and os.path.exists(os.path.join(MIND_TRAIN, 'news.tsv')):
    tn, tb = _count(os.path.join(MIND_TRAIN, 'news.tsv')), _count(os.path.join(MIND_TRAIN, 'behaviors.tsv'))
    dn, db = _count(os.path.join(MIND_DEV, 'news.tsv')), _count(os.path.join(MIND_DEV, 'behaviors.tsv'))
    print('✅ ĐÃ CÓ MIND THẬT — benchmark sẽ chạy trên DỮ LIỆU THẬT')
    print('   train:', MIND_TRAIN)
    print(f'      news.tsv = {tn:,} dòng | behaviors.tsv = {tb:,} dòng')
    print('   dev  :', MIND_DEV)
    print(f'      news.tsv = {dn:,} dòng | behaviors.tsv = {db:,} dòng')
    print('   (MIND-small chuẩn ~ news 51k, train ~156k, dev ~73k dòng)')
    with open(os.path.join(MIND_TRAIN, 'news.tsv'), encoding='utf-8') as f:
        print('   Ví dụ news :', f.readline().strip()[:90])
    IS_REAL = True
else:
    print('⚠️  CHƯA CÓ MIND — notebook sẽ chạy SYNTHETIC (KHÔNG phải data thật).')
    print('   Bật Settings > Internet = On (để ô 2a tự tải), hoặc Add Input > Datasets dataset MIND.')
    IS_REAL = False
print('=' * 60)

### 3. (Tùy chọn) Tính embedding BGE thật cho tin — cần Internet On + đã gắn MIND

In [ ]:
import os, subprocess, sys
if USE_BGE and MIND_TRAIN and os.path.exists(os.path.join(MIND_TRAIN, 'news.tsv')):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)
    # gộp news của cả 2 split để phủ hết id
    subprocess.run([sys.executable, 'llm_embed.py',
                    '--news', os.path.join(MIND_TRAIN, 'news.tsv'),
                    '--out', 'news_emb.npz', '--model', BGE_MODEL], check=True)
    NEWS_EMB = 'news_emb.npz'
    print('BGE embeddings ->', NEWS_EMB)
else:
    print('Bỏ qua BGE (USE_BGE=False hoặc chưa gắn MIND). llmenc sẽ dùng embedding placeholder.')

### 4. Chạy benchmark (tự fallback synthetic nếu chưa gắn MIND)

In [ ]:
import os, subprocess, sys
cmd = [sys.executable, 'bench.py', '--epochs', str(EPOCHS), '--dim', str(DIM),
       '--eval-batch', str(EVAL_BATCH), '--models', *MODELS, '--out', 'results/kaggle']
if MIND_TRAIN and os.path.exists(os.path.join(MIND_TRAIN, 'news.tsv')):
    cmd += ['--source', 'mind', '--mind-train', MIND_TRAIN, '--mind-dev', MIND_DEV]
    if NEWS_EMB and os.path.exists(NEWS_EMB):
        cmd += ['--news-emb', NEWS_EMB]
    print('>> RUN on REAL MIND')
else:
    cmd += ['--source', 'synthetic']
    print('>> MIND chưa gắn -> RUN SYNTHETIC demo')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

### 5. Xem kết quả

In [ ]:
print(open('results/kaggle/results.md').read())
# JSON chi tiết: results/kaggle/results.json

---
**Ghi chú**
- `--models core` = nrms, naml, fastformer, caum, lightgcn, llmenc, supermodel (mặc định gọn).
- Muốn `llmenc`/`supermodel` **mạnh trên MIND**: đặt `USE_BGE=True` + Internet On + gắn MIND.
- MIND-large nặng hơn: tăng `EVAL_BATCH` nếu GPU khỏe, hoặc chạy MIND-small trước.
- Hướng mở rộng (SID-News: Semantic-ID generative) sẽ là module thêm, không phá code này.